# Tier 2 Performance Benchmark — Pipeline Optimization

**Purpose:** Measure pipeline performance after Tier 2 optimizations and compare against Tier 1 baseline.

## Tier 1 Baseline (pre-Tier 2)

| Difficulty | Baseline (pre-T1) | Target After T2 |
|---|---|---|
| recall | 125.1s | similar |
| comprehension | 122.7s | similar |
| application | 539.9s | ~300s |
| high_application | 1290.8s | ≤ 400s |
| **TOTAL** | **2078.5s** | **≤ 900s** |

## Tier 2 Optimizations Applied

- **T-OPT-2.1:** Parse-only retry (cache raw text, retry parsing before re-generating) — saves ~30-60s per parse failure
- **T-OPT-2.3:** Parallel micro-batch generation (split large batches into 2 parallel generation calls)
- **T-OPT-2.2:** SKIPPED — JSON mode has infinite loop bugs on Gemini 3 Flash with strict schemas

## Targets

- `high_application` ≤ 400s (was 1290.8s baseline)
- Total pipeline ≤ 900s (was 2078.5s baseline)
- 100% delivery maintained (40/40 items)

## Section 1: Setup & Imports

In [1]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
os.environ.setdefault('ENV', 'develop')

from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope, GameContentResponse
)
from src.graph.builder import compile_graph
from src.graph.nodes.math_agent import (
    MICRO_BATCH_THRESHOLD, MAX_PARSE_ONLY_RETRIES,
    MAX_GENERATION_RETRIES, MAX_PARSE_RETRIES,
)

print('Imports OK')
print(f'MICRO_BATCH_THRESHOLD = {MICRO_BATCH_THRESHOLD}')
print(f'MAX_PARSE_ONLY_RETRIES = {MAX_PARSE_ONLY_RETRIES}')
print(f'MAX_GENERATION_RETRIES = {MAX_GENERATION_RETRIES}')
print(f'MAX_PARSE_RETRIES = {MAX_PARSE_RETRIES}')

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
MICRO_BATCH_THRESHOLD = 12
MAX_PARSE_ONLY_RETRIES = 2
MAX_GENERATION_RETRIES = 2
MAX_PARSE_RETRIES = 2


## Section 2: E2E Runner

In [2]:
NUM_QUESTIONS = 10
TOPIC = "Đạo hàm và ứng dụng"

# Baseline numbers (pre-Tier 1)
BASELINE_PRE_T1 = {
    "recall": {"time": 125.1, "iterations": 1},
    "comprehension": {"time": 122.7, "iterations": 1},
    "application": {"time": 539.9, "iterations": 2},
    "high_application": {"time": 1290.8, "iterations": 4},
}
BASELINE_TOTAL = sum(b["time"] for b in BASELINE_PRE_T1.values())


async def run_e2e(difficulty: str):
    """Run pipeline end-to-end and return (num_delivered, elapsed, iterations, errors)."""
    req = GenerationRequest(
        user_id="perf_benchmark_tier2",
        topic=TOPIC,
        game_types=[GameType.QUIZ],
        num_questions=NUM_QUESTIONS,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope.SYSTEM,
    )
    app = compile_graph()
    initial_state = {
        "request": req,
        "doc_scope": req.doc_scope.value,
        "iteration_count": 0,
        "rejected_items": [],
        "reviewed_items": [],
        "errors": [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0

    output: GameContentResponse | None = result.get("final_output")
    delivered = len(output.content.quiz) if output else 0
    iterations = result.get("iteration_count", 0)
    errors = result.get("errors", [])
    return delivered, elapsed, iterations, errors


print(f"Runner ready — Topic: {TOPIC} | {NUM_QUESTIONS} questions per level")

Runner ready — Topic: Đạo hàm và ứng dụng | 10 questions per level


## Section 3: Run Benchmark (All 4 Difficulty Levels)

In [3]:
print("=" * 70)
print("TIER 2 PERFORMANCE BENCHMARK")
print(f"Topic: {TOPIC} | Requested: {NUM_QUESTIONS} quiz per level")
print("=" * 70)

e2e_results = {}
for diff in ["recall", "comprehension", "application", "high_application"]:
    print(f"\n--- Running {diff} ---")
    delivered, elapsed, iters, errors = await run_e2e(diff)
    pct = delivered / NUM_QUESTIONS * 100
    status = "✅" if delivered >= NUM_QUESTIONS else "❌"
    baseline_t = BASELINE_PRE_T1[diff]["time"]
    speedup = (baseline_t - elapsed) / baseline_t * 100
    print(f"  {status} Delivered: {delivered}/{NUM_QUESTIONS} ({pct:.0f}%) in {elapsed:.1f}s ({iters} iter)")
    print(f"  ⏱️  Baseline: {baseline_t:.1f}s → Now: {elapsed:.1f}s ({speedup:+.1f}%)")
    if errors:
        print(f"  ⚠️ Errors: {errors}")
    e2e_results[diff] = {
        "delivered": delivered,
        "requested": NUM_QUESTIONS,
        "pct": pct,
        "elapsed": elapsed,
        "iterations": iters,
        "passed": delivered >= NUM_QUESTIONS,
        "baseline": baseline_t,
        "speedup_pct": speedup,
    }

TIER 2 PERFORMANCE BENCHMARK
Topic: Đạo hàm và ứng dụng | Requested: 10 quiz per level

--- Running recall ---
2026-04-10 15:38:33 [info     ] creating_graph                
2026-04-10 15:38:33 [info     ] graph_created                 
2026-04-10 15:38:33 [info     ] graph_compiled_without_checkpointer
2026-04-10 15:38:33 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm và ứng dụng' user_id=perf_benchmark_tier2
2026-04-10 15:38:33 [info     ] creating_review_llm            location=global max_output_tokens=10 model=gemini-3-flash-preview temperature=0
2026-04-10 15:38:33 [info     ] rate_limiter_init_bucket       capacity=5.0 refill_rate=1.0 rpm=60
2026-04-10 15:38:33 [info     ] rate_limiter_init_semaphore    max_concurrent=10
2026-04-10 15:38:34 [warning  ] unsupported_content_type_defaulting_to_math detected=
2026-04-10 15:38:34 [info     ] supervisor_classified          content_type=math
2026-04-10 15:38:34 [info     ] routing_to_agent               conte

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-04-10 15:38:38 [debug    ] retrieved_documents            count=11
2026-04-10 15:38:38 [info     ] context_retrieved              chunks=11 sources=11
2026-04-10 15:38:38 [info     ] math_agent_context_retrieved   context_chunks=11
2026-04-10 15:38:38 [info     ] creating_generation_llm        location=global max_output_tokens=8192 model=gemini-3-flash-preview temperature=0.7
2026-04-10 15:38:38 [info     ] math_agent_micro_batch_generation batch_a=6 batch_b=7 total=13
2026-04-10 15:38:54 [info     ] math_agent_code_execution_done batch_size=6 code_traces_count=2 generation_attempt=1 text_length=2382
2026-04-10 15:38:54 [info     ] creating_generation_llm        location=global max_output_tokens=8192 model=gemini-3-flash-preview temperature=0.3


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-04-10 15:39:00 [info     ] math_agent_code_execution_done batch_size=7 code_traces_count=2 generation_attempt=1 text_length=3776


...

2026-04-10 15:43:40 [info     ] math_agent_code_execution_done batch_size=7 code_traces_count=2 generation_attempt=1 text_length=3846
2026-04-10 15:44:01 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=7 parse_attempt=1
2026-04-10 15:44:07 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=7 parse_attempt=1
2026-04-10 15:44:07 [info     ] math_agent_generated           items_count=14
2026-04-10 15:44:07 [info     ] reviewer_reviewing             items_count=14 iteration=0
2026-04-10 15:44:34 [error    ] reviewer_failed                error='Failed to parse ReviewBatch from completion {"reviews": [{"item_index": 0, "passed": true, "score": 1.0, "feedback": "Correct calculation and clear explanation."}, {"item_index": 1, "passed": true, "score": 1.0, "feedback": "Correct calculation and clear explanation."}, {"item_index": 2, "passed": true, "score": 1.0, "feedback": "Correct calculation and clear explanation."}, {}]}. Got: 4 validation errors for

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-04-10 15:45:06 [debug    ] retrieved_documents            count=30
2026-04-10 15:45:06 [info     ] context_retrieved              chunks=30 sources=30
2026-04-10 15:45:06 [info     ] math_agent_context_retrieved   context_chunks=30
2026-04-10 15:45:06 [info     ] math_agent_micro_batch_generation batch_a=12 batch_b=13 total=25
2026-04-10 15:45:53 [info     ] math_agent_code_execution_done batch_size=13 code_traces_count=2 generation_attempt=1 text_length=11764


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-04-10 15:46:00 [info     ] math_agent_code_execution_done batch_size=12 code_traces_count=2 generation_attempt=1 text_length=8418
2026-04-10 15:46:15 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-04-10 15:46:24 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=5
2026-04-10 15:46:27 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-10 15:46:27 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=12 parse_attempt=1
2026-04-10 15:46:31 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-10 15:46:31 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=13 parse_attempt=1
2026-04-10 15:46:31 [info     ] math_agent_generated           items_count=25
2026-04-10 15:46:31 [info     ] reviewer_reviewing             items_count=25 iteration=0
2026-04-10 15:46:57 [error    ] reviewer_failed                error='Failed to parse ReviewBatch from completion {"reviews

## Section 4: Verdict & Comparison

In [4]:
print("\n" + "=" * 85)
print("TIER 2 PERFORMANCE BENCHMARK — RESULTS")
print("=" * 85)
print(f"{'Difficulty':<20} {'Delivered':>9} {'Rate':>7} {'Baseline':>9} {'Now':>8} {'Speedup':>9} {'Status':>8}")
print("-" * 85)

all_passed = True
for diff, r in e2e_results.items():
    status = "✅ PASS" if r["passed"] else "❌ FAIL"
    if not r["passed"]:
        all_passed = False
    print(
        f"{diff:<20} {r['delivered']:>5}/{r['requested']:<3} {r['pct']:>6.0f}% "
        f"{r['baseline']:>8.1f}s {r['elapsed']:>7.1f}s {r['speedup_pct']:>+8.1f}% {status:>8}"
    )

print("-" * 85)
total_d = sum(r["delivered"] for r in e2e_results.values())
total_r = sum(r["requested"] for r in e2e_results.values())
total_pct = total_d / total_r * 100 if total_r else 0
total_time = sum(r["elapsed"] for r in e2e_results.values())
total_speedup = (BASELINE_TOTAL - total_time) / BASELINE_TOTAL * 100
print(
    f"{'TOTAL':<20} {total_d:>5}/{total_r:<3} {total_pct:>6.0f}% "
    f"{BASELINE_TOTAL:>8.1f}s {total_time:>7.1f}s {total_speedup:>+8.1f}%"
)

print("\n" + "=" * 85)
print("PERFORMANCE TARGETS (Tier 2)")
print("=" * 85)

# Target 1: 100% delivery maintained
delivery_ok = all_passed
print(f"  {'✅' if delivery_ok else '❌'} 100% delivery: {total_d}/{total_r} ({total_pct:.0f}%)")

# Target 2: high_application <= 400s (Tier 2 target)
ha_time = e2e_results["high_application"]["elapsed"]
ha_ok = ha_time <= 400
print(f"  {'✅' if ha_ok else '⚠️'} high_application ≤ 400s: {ha_time:.1f}s")

# Target 3: total <= 900s (Tier 2 target)
total_ok = total_time <= 900
print(f"  {'✅' if total_ok else '⚠️'} Total ≤ 900s: {total_time:.1f}s")

# Tier 1 target check (regression)
ha_t1_ok = ha_time <= 800
total_t1_ok = total_time <= 1400
print(f"  {'✅' if ha_t1_ok else '❌'} [Tier 1] high_application ≤ 800s: {ha_time:.1f}s")
print(f"  {'✅' if total_t1_ok else '❌'} [Tier 1] Total ≤ 1400s: {total_time:.1f}s")

print("\n" + "=" * 85)
if delivery_ok:
    print(f"🎯 100% delivery maintained | Total speedup: {total_speedup:+.1f}% ({BASELINE_TOTAL:.0f}s → {total_time:.0f}s)")
else:
    failed = [d for d, r in e2e_results.items() if not r["passed"]]
    print(f"❌ DELIVERY REGRESSION — Failed: {', '.join(failed)}")

print("\nOptimization details:")
print(f"  T-OPT-2.1: Parse-only retry (MAX_PARSE_ONLY_RETRIES={MAX_PARSE_ONLY_RETRIES})")
print(f"  T-OPT-2.3: Parallel micro-batch gen (MICRO_BATCH_THRESHOLD={MICRO_BATCH_THRESHOLD})")
print(f"  T-OPT-2.2: SKIPPED (JSON mode infinite loop risk)")
print("=" * 85)


TIER 2 PERFORMANCE BENCHMARK — RESULTS
Difficulty           Delivered    Rate  Baseline      Now   Speedup   Status
-------------------------------------------------------------------------------------
recall                  10/10     100%    125.1s    90.1s    +28.0%   ✅ PASS
comprehension           10/10     100%    122.7s   185.0s    -50.8%   ✅ PASS
application             10/10     100%    539.9s   114.4s    +78.8%   ✅ PASS
high_application        10/10     100%   1290.8s   143.7s    +88.9%   ✅ PASS
-------------------------------------------------------------------------------------
TOTAL                   40/40     100%   2078.5s   533.2s    +74.3%

PERFORMANCE TARGETS (Tier 2)
  ✅ 100% delivery: 40/40 (100%)
  ✅ high_application ≤ 400s: 143.7s
  ✅ Total ≤ 900s: 533.2s
  ✅ [Tier 1] high_application ≤ 800s: 143.7s
  ✅ [Tier 1] Total ≤ 1400s: 533.2s

🎯 100% delivery maintained | Total speedup: +74.3% (2078s → 533s)

Optimization details:
  T-OPT-2.1: Parse-only retry (MAX_PARSE_O